# NetForge-MARL — Fine-tune Llama-3-8B as a Blue SOC operator (LoRA + PPO)

Flagship demonstrator for Phase 8 of the [NetForge roadmap](../ROADMAP.md). Designed for a single 24 GB consumer GPU (Colab Pro T4 / A10 / RTX 3090).

**Wall-clock target:** ≤ 4 hours end-to-end on a single GPU.

## What this notebook does
1. Installs NetForge with the `[finetune]` extra (4-bit + LoRA + trl).
2. Loads `meta-llama/Meta-Llama-3-8B-Instruct` in 4-bit with LoRA adapters.
3. Wires the env to the model via `LMPolicyAdapter` — every PPO step is one real episode tick.
4. Trains for 2,000 PPO steps with the SIEM prompt as the query and `ACTION ... TARGET ...` strings as the response.
5. Evaluates the fine-tuned adapter against the base model on held-out scenarios; pushes the LoRA adapter to the Hub.

In [ ]:
!pip install -q 'git+https://github.com/reforcemind/NetForge_RL'
!pip install -q 'trl>=0.9.0' 'transformers>=4.44.0' 'peft>=0.12.0' 'bitsandbytes>=0.43.0' 'accelerate>=0.33.0'

In [ ]:
from huggingface_hub import notebook_login
notebook_login()  # Llama-3 is gated; accept the licence on HF first.

In [ ]:
from netforge_rl.semantic.finetune.ppo_recipe import main
main('netforge_rl/semantic/finetune/configs/llama3_8b_lora.yaml')

## Evaluate base vs fine-tuned

Run 50 episodes each and compare Mean Time To Containment + final compromised host count.

In [ ]:
from netforge_rl.environment.parallel_env import NetForgeRLEnv
from netforge_rl.semantic import run_episode, append_result, summarize
from netforge_rl.semantic.clients import MockLLMClient  # swap for an HF client wrapping your tuned model

env = NetForgeRLEnv({'scenario_type': 'ransomware', 'max_ticks': 200})
for i in range(50):
    res = run_episode(env, {'blue_dmz': MockLLMClient(seed=i)}, max_steps=200, seed=i)
    append_result('leaderboard/llama3_blue.json', res)
summarize('leaderboard/llama3_blue.json')

## Push to the Hub

```python
from huggingface_hub import HfApi
HfApi().upload_folder(
    folder_path='./runs/netforge-llama3-blue-v1/step-2000',
    repo_id='reforcemind/netforge-llama3-blue-v1',
    repo_type='model',
)
```